In [1]:
import os
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools import agent_tool
from google.genai import types
from mysql import connector
from sentence_transformers import SentenceTransformer, util
from dotenv import load_dotenv
load_dotenv()

c:\Users\ritur\.conda\envs\multi_agent_base\Lib\site-packages\google\cloud\aiplatform\models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils
c:\Users\ritur\.conda\envs\multi_agent_base\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

#### Initializing the LLM

In [ ]:
MODEL_GPT = "openai/gpt-4o-mini"

llm = LiteLlm(model=MODEL_GPT)
llm.llm_client.completion(model=llm.model, messages=[{"role": "user", "content":"Are you ready"}],
                                temperature=0.1,
                                tools=[])
print("Open AI is ready")

Open AI is ready


#### Creating the tools

In [3]:
INSERT_QUERY = "INSERT INTO tickets (id, username, issue_create_dt, issue_desc, issue_status) VALUES (%s, %s, sysdate(), %s, 'OP')"

SEARCH_EXISTING_ID_QUERY = "SELECT * FROM tickets WHERE id = %s"

TICKET_STATUS_QUERY = "SELECT * FROM tickets WHERE id = %s AND username = %s"

def create_ticket(username: str, ticketId: int, user_message: str) -> dict:
    returnDict = None
    mydb = connector.connect(
        host="127.0.0.1",
        port=3306,
        user="root",
        password="12345678",
        database="ticketDb"
    )
    # Verify connection
    if mydb.is_connected():
        cursor = mydb.cursor()
        values = (ticketId, username, user_message)
        cursor.execute(INSERT_QUERY, values)
        mydb.commit()
        cursor.close()
        mydb.close()
        returnDict = {"status": "success", "query_results": [{'reply': 'Row inserted.'}]}
    else:
        returnDict = {"status": "error", "query_results": [{'reply': 'DB connection error.'}]}
    
    return returnDict

def search_ticket_with_id(username: str, ticketId: int) -> dict:
    returnDict = None
    mydb = connector.connect(
        host="127.0.0.1",
        port=3306,
        user="root",
        password="12345678",
        database="ticketDb"
    )
    # Verify connection
    if mydb.is_connected():
        cursor = mydb.cursor()
        values = (ticketId, username)
        cursor.execute(TICKET_STATUS_QUERY, values)
        results = cursor.fetchall()
        if results is not []:
            ticket_details = results[0]
            ticket_status = get_ticket_status(ticket_details[4])
            returnDict = {"status": "success", "query_results": [{'reply': f'Found. Status is {ticket_status}'}]}
        else:
            returnDict = {"status": "success", "query_results": [{'reply': 'Not found.'}]}
        cursor.close()
        mydb.close()
    else:
        returnDict = {"status": "error", "query_results": [{'reply': 'DB connection error.'}]}
    return returnDict

def check_ticket_id(ticketId: int) -> dict:
    returnDict = None
    mydb = connector.connect(
        host="127.0.0.1",
        port=3306,
        user="root",
        password="12345678",
        database="ticketDb"
    )
    # Verify connection
    if mydb.is_connected():
        cursor = mydb.cursor()
        values = (ticketId)
        cursor.execute(SEARCH_EXISTING_ID_QUERY, values)
        results = cursor.fetchall()
        if results is not []:
            returnDict = {"status": "success", "query_results": [{'reply': 'Found existing ticket.'}]}
        else:
            returnDict = {"status": "success", "query_results": [{'reply': 'Not found.'}]}
        cursor.close()
        mydb.close()
    else:
        returnDict = {"status": "error", "query_results": [{'reply': 'DB connection error.'}]}
    return returnDict

def get_ticket_status(status_code: str) -> str:
    status_str = ""
    match status_code:
        case 'OP':
            status_str = 'Open'
        case 'IP':
            status_str = 'In Progress'
        case 'RS':
            status_str = 'Resolved'
        case 'BL':
            status_str = 'Blocked'
        case _:
            status_str = 'Not available'
    return status_str

In [4]:
MODEL_TXT_SIMILARITY = "all-MiniLM-L6-v2"

OPEN_TICKETS_WITH_USER_QUERY = "SELECT * FROM tickets WHERE username = %s and issue_status = 'OP'"

def search_ticket_with_message(username: str, user_message: str) -> dict:
    returnDict = None
    similarity_model = SentenceTransformer(MODEL_TXT_SIMILARITY)
    mydb = connector.connect(
        host="127.0.0.1",
        port=3306,
        user="root",
        password="12345678",
        database="ticketDb"
    )
    # Verify connection
    if mydb.is_connected():
        cursor = mydb.cursor()
        values = (username,)
        cursor.execute(OPEN_TICKETS_WITH_USER_QUERY, values)
        results = cursor.fetchall()
        if results is not []:
            issue_desc_list = [data[3] for data in results]
            target_embedding = similarity_model.encode(user_message, convert_to_tensor=True)
            list_embeddings = similarity_model.encode(issue_desc_list, convert_to_tensor=True)
            cosine_scores = [score.item() for score in util.cos_sim(target_embedding, list_embeddings)[0]]
            highest_cosine_score = max(cosine_scores)
            target_issue = None
            if highest_cosine_score > 0.8:
                target_issue_desc = issue_desc_list[cosine_scores.index(highest_cosine_score)]
                target_idx = next((i for i, item in enumerate(results) if item[3] == target_issue_desc), -1)
                if target_idx != -1:
                    target_issue = results[target_idx]
            if target_issue is not None:
                returnDict = {"status": "success", "query_results": [{'reply': f'Found. Ticket Id is {target_issue[0]} and status is open.'}]}
            else:
                returnDict = {"status": "success", "query_results": [{'reply': 'Not found.'}]}
        else:
            returnDict = {"status": "success", "query_results": [{'reply': 'Not found.'}]}
        cursor.close()
        mydb.close()
    else:
        returnDict = {"status": "error", "query_results": [{'reply': 'DB connection error.'}]}
    return returnDict


#### Testing the tools

In [5]:
search_ticket_with_id("ELE23HH94jo", 999999)

{'status': 'success', 'query_results': [{'reply': 'Found. Status is Open'}]}

In [6]:
search_ticket_with_message("ELE23HH94jo", "I still have not received my replacement debit card.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2221.30it/s]


{'status': 'success',
 'query_results': [{'reply': 'Found. Ticket Id is 999999 and status is open.'}]}

In [2]:
# Load a pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Define your target sentence and array of sentences
target_sentence = "The quick brown fox jumps over the lazy dog."
sentences_array = [
    "A fast brown fox leaps over a sleeping dog.",
    "The weather is lovely today.",
    "Quick foxes are jumping over lazy dogs.",
    "I am learning how to code in Python."
]

# Encode the sentences to get their embeddings
target_embedding = model.encode(target_sentence, convert_to_tensor=True)
array_embeddings = model.encode(sentences_array, convert_to_tensor=True)

# Compute cosine similarity
cosine_scores = util.cos_sim(target_embedding, array_embeddings)[0]

# Pair the sentences with their scores and rank them
results = []
for sentence, score in zip(sentences_array, cosine_scores):
    results.append({"sentence": sentence, "score": score.item()})

# Sort by highest similarity score
results_sorted = sorted(results, key=lambda x: x["score"], reverse=True)

# Display results
for res in results_sorted:
    print(f"Score: {res['score']:.4f} | Sentence: {res['sentence']}")


c:\Users\ritur\.conda\envs\multi_agent_base\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ritur\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6282.92it/s]


Score: 0.8065 | Sentence: A fast brown fox leaps over a sleeping dog.
Score: 0.8038 | Sentence: Quick foxes are jumping over lazy dogs.
Score: 0.0766 | Sentence: The weather is lovely today.
Score: 0.0588 | Sentence: I am learning how to code in Python.


#### Creating a runner method

In [7]:
async def agent_runner(agent: Agent, content: types.Content) -> str:
    """Create and return an AgentCaller instance for the given agent."""
    app_name = agent.name + "_app_1"
    user_id = agent.name + "_user_1"
    session_id = agent.name + "_session_01"

    session_service = InMemorySessionService()

    await session_service.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id
    )

    runner = Runner(
        agent=agent,
        app_name=app_name,
        session_service=session_service
    )

    response_text = "Agent did not produce a final response"

    verbose= False

    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
        if verbose:
            print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

        if event.is_final_response():
            if event.content and event.content.parts:
                response_text = event.content.parts[0].text ## # Assuming text response in the first part

            elif event.actions and event.actions.escalate:
                response_text = f"Agent escalated: {event.error_message or 'No specific message provided.'}"
    
    return response_text

#### Creating an intent classifier agent

In [10]:
intent_agent = Agent(
    name= "intent_agent_v1",
    model=llm,
    description="Classifies the user query based on intent",
    instruction="""You are a helpful Banking Customer Support intent classifier, chatting with a user. 
                
                Based on the user message, categorize the user's intent into one of the below three categories:
                Positive Feedback, Negative Feedback, Query

                If the intent is Positive Feedback, return 0.
                If the intent is Negative Feedback, return 1.
                If the intent is Query, return 2.
                If the intent is not classified as any of the above three categories or if the message is not related to banking, return 3.
                
                Strictly include only a number denoting the intent category in the response.
                """,
    tools=[]
)

#### Creating the feedback agent

In [11]:
feedback_agent = Agent(
    name= "feedback_agent_v1",
    model=llm,
    description="Generates appropriate response to the user feedback",
    instruction="""You are a helpful feedback response agent, chatting with a user to come up with an appropriate response.
                Include username in all the generated responses. 
                
                Use the provided user message intent to decide action code. 
                If user message intent is 0, action code is 0.
                If user message intent is 1, action code is 1. 
                
                If the action code is 0, generate a warm, personalized thank-you message for response. 
                
                Use the below response format:
                [thank-you message]

                If the action is 1, generate an personalized empathetic message. 
                Use the search_ticket_with_message tool to check if any open tickets exists with same description for the user.
                If yes, return ticket details with current status along with the generated message.
                If no, generate a unique 6-digit ticket id. 
                Use the check_ticket_id tool to check if a ticket already exists with the generated ticketId.
                If yes, continue with the new ticket id generation and check_ticket_id tool usage until no existing tickets are found.
                If no existing tickets are found, use the create_ticket tool to add a new ticket.
                If the response from the create_ticket tool is successful, return ticket details along with the generated message.
                
                Use the below response format:
                [empathetic message].[ticket details message]

                """,
    tools=[create_ticket, search_ticket_with_message, check_ticket_id, get_ticket_status]
)

#### Creating the query handler agent

In [12]:
query_agent = Agent(
    name= "query_agent_v1",
    model=llm,
    description="Generates appropriate response to the user query",
    instruction="""You are a helpful query response agent, chatting with a user to come up with an appropriate response.
                Include username in all the generated responses. 
                
                Use the provided user message to get the provided ticketId. Use the search_ticket_with_id tool to get the ticket details.
                Generate an appropriate response message containing ticket details and current status.

                If ticketId is not available in the user message, generate a response message stating that only ticket lookup functionality is supported for now.
                """,
    tools=[search_ticket_with_id]
)

#### Running the agents

In [13]:
chat_username = "ELE23HH94jo"

pos_user_message = "Thanks for resolving my credit card issue."
neg_user_message = "I still have not received my replacement debit card."
real_query_message = "Could you check the status of ticket 999999?"
not_query_message = "Hi, I woke up and eat breakfast. Happy days!!"

intent_content = types.Content(role='user', parts=[types.Part(text=f"Username: {chat_username}"),
                                                  types.Part(text=f"User message: {neg_user_message}")])


#### Using if-else logic for agent call

In [14]:
intent_response_text = await agent_runner(intent_agent, intent_content)
next_agent_response = None
print(f" Intent Response -- {intent_response_text} ")

if intent_response_text == "0" or intent_response_text == "1":
    feedback_content = types.Content(role='user', parts=[types.Part(text=f"Username: {chat_username}"),
                                                          types.Part(text=f"User message: {neg_user_message}"),
                                                          types.Part(text=f"User message intent: {intent_response_text}")])
    next_agent_response = await agent_runner(feedback_agent, feedback_content)
else:
    query_content = types.Content(role='user', parts=[types.Part(text=f"Username: {chat_username}"),
                                                    types.Part(text=f"User message: {real_query_message}"),
                                                    types.Part(text=f"User message intent: {intent_response_text}")])
    next_agent_response = await agent_runner(query_agent, query_content)

print(f" Next Agent Response -- {next_agent_response} ")

 Intent Response -- 1 


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6003.35it/s]


 Next Agent Response -- I understand how frustrating it can be to wait for your replacement debit card, ELE23HH94jo. It seems there is an open ticket regarding your issue. 

**Ticket Details:**  
- **Ticket ID:** 999999  
- **Current Status:** Open  

Please rest assured that the team is working on it and will get back to you as soon as possible. If you need further assistance, feel free to reach out! 


#### Using a coordinator agent to determine which agent to call

In [ ]:
agent_content = types.Content(role='user', parts=[types.Part(text=f"Username: {chat_username}"),
                                                  types.Part(text=f"User message: {neg_user_message}")])

intent_tool = agent_tool.AgentTool(agent=intent_agent)

coordinator_agent = Agent(
    name= "coordinator_agent_v1",
    model=llm,
    description="Main coordinator.",
    instruction="""You are a helpful Banking Customer Support Coordinator assistant. 
                Determine the intent of the user message using the intent_tool.
                Based on the intent, delegate the tasks to the other sub-agents.
                If the intent is 0 or 1, delegate the task to feedback_agent.
                Else, delegate the task to query_agent.
                """,
    tools=[intent_tool],
    sub_agents=[feedback_agent, query_agent]
)

In [10]:
await agent_runner(coordinator_agent, agent_content)

<<< Agent Response -- I'm really sorry to hear that your debit card replacement still hasn't arrived, ELE23HH94jo. I understand how frustrating this must be for you. I've created a ticket to track this issue for you.

**Ticket Details:**
- **Ticket ID:** 218849
- **Status:** Open

We’ll work on this and get back to you as soon as possible. Thank you for your patience!


"I'm really sorry to hear that your debit card replacement still hasn't arrived, ELE23HH94jo. I understand how frustrating this must be for you. I've created a ticket to track this issue for you.\n\n**Ticket Details:**\n- **Ticket ID:** 218849\n- **Status:** Open\n\nWe’ll work on this and get back to you as soon as possible. Thank you for your patience!"

#### Streamlit UI

In [1]:
import streamlit as st
import random
import time


# Streamed response emulator
def response_generator():
    response = random.choice(
        [
            "Hello there! How can I assist you today?",
            "Hi, human! Is there anything I can help you with?",
            "Do you need help?",
        ]
    )
    for word in response.split():
        yield word + " "
        time.sleep(0.05)


st.title("Simple chat")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Accept user input
if prompt := st.chat_input("What is up?"):
    # Add user message to chat history
    st.session_state.messages.append({"role": "user", "content": prompt})
    # Display user message in chat message container
    with st.chat_message("user"):
        st.markdown(prompt)

    # Display assistant response in chat message container
    with st.chat_message("assistant"):
        response = st.write_stream(response_generator())
    # Add assistant response to chat history
    st.session_state.messages.append({"role": "assistant", "content": response})

2026-05-31 14:21:33.541 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-31 14:21:34.214 
  command:

    streamlit run c:\Users\ritur\.conda\envs\multi_agent_base\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-31 14:21:34.216 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-31 14:21:34.216 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-31 14:21:34.217 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-31 14:21:34.217 Session state does not function when running a script without `streamlit run`
2026-05-31 14:21:34.218 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-31 14:21:34.218 Thread 'MainThread': missing